# 05. Изменчивость W70/F30

Цель — отделить нормальную вариабельность отношения двух расходов одной фракции от остановов, кодовых значений и переходов. Границы ниже являются observed ranges, а не технологическими нормативами.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.express as px
from IPython.display import display

HERE = Path.cwd().resolve()
ROOT = next((p for p in [HERE, *HERE.parents] if (p / 'data' / 'avt_tags.csv').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Не найден data/avt_tags.csv. Запустите ноутбук внутри проекта Нефтекод.')
EDA_DIR = ROOT / 'eda'
DATA_DIR = ROOT / 'data'
ARTIFACTS = EDA_DIR / 'artifacts'
ARTIFACTS.mkdir(parents=True, exist_ok=True)
print('Корень проекта:', ROOT)
print('Файл данных:', DATA_DIR / 'avt_tags.csv')
raw = pd.read_csv(DATA_DIR / 'avt_tags.csv', parse_dates=['date'], low_memory=False)
raw = raw.drop(columns=[c for c in raw if c.startswith('Unnamed:')], errors='ignore')
raw['ratio'] = raw.W70 / raw.F30
raw.shape

(189217, 73)

## Маска рабочего диапазона

Используется диагностическая маска `10 ≤ F30 ≤ 200` и `W70 > 0`. Она удаляет отрицательные/околонулевые расходы и кодовые плато 251/307. Это не паспортный operating limit.

In [2]:
valid_mask = raw.F30.between(10, 200) & raw.W70.gt(0)
data = raw.loc[valid_mask].copy()
coverage = pd.DataFrame([{
    'all_rows': len(raw), 'working_rows': len(data),
    'working_share_pct': 100*len(data)/len(raw),
    'excluded_rows': len(raw)-len(data),
}])
display(coverage)
coverage.to_csv(ARTIFACTS / 'w70_f30_ratio_coverage.csv', index=False)

,all_rows,working_rows,working_share_pct,excluded_rows
0,189217,181527,95.935883,7690


## Распределение отношения

In [3]:
quantile_levels = [.001,.005,.01,.025,.05,.25,.5,.75,.95,.975,.99,.995,.999]
quantiles = data.ratio.quantile(quantile_levels).rename_axis('quantile').reset_index(name='ratio')
median = data.ratio.median()
mad = (data.ratio-median).abs().median()
p01, p99 = data.ratio.quantile([.01,.99])
central = data[data.ratio.between(p01,p99)].copy()
summary = pd.DataFrame([{
    'n':len(data), 'mean':data.ratio.mean(), 'std':data.ratio.std(),
    'coefficient_of_variation_pct':100*data.ratio.std()/data.ratio.mean(),
    'median':median, 'MAD':mad, 'robust_sigma':1.4826*mad,
    'robust_coefficient_of_variation_pct':100*1.4826*mad/median,
    'p01':p01, 'p05':data.ratio.quantile(.05), 'p95':data.ratio.quantile(.95), 'p99':p99,
    'p05_p95_width_pct_of_median':100*(data.ratio.quantile(.95)-data.ratio.quantile(.05))/median,
    'p01_p99_width_pct_of_median':100*(p99-p01)/median,
}])
display(summary, quantiles)
summary.to_csv(ARTIFACTS / 'w70_f30_ratio_summary.csv', index=False)
quantiles.to_csv(ARTIFACTS / 'w70_f30_ratio_quantiles.csv', index=False)
px.histogram(data.query('ratio >= @p01 and ratio <= @p99'), x='ratio', nbins=120,
             title='W70/F30: центральные 98% рабочих наблюдений').show()

,n,mean,std,coefficient_of_variation_pct,median,MAD,robust_sigma,robust_coefficient_of_variation_pct,p01,p05,p95,p99,p05_p95_width_pct_of_median,p01_p99_width_pct_of_median
0,181527,0.78115,0.008372,1.071694,0.781246,0.003145,0.004662,0.596767,0.76721,0.773167,0.788992,0.792493,2.02563,3.236267


,quantile,ratio
0,0.001,0.757277
1,0.005,0.763999
2,0.010,0.767210
3,0.025,0.771342
4,0.050,0.773167
5,0.250,0.778208
6,0.500,0.781246
7,0.750,0.784495
8,0.950,0.788992
9,0.975,0.790573


ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

In [ ]:
bands = []
for relative_band in [.005,.01,.02,.05]:
    bands.append({'relative_band_pct':100*relative_band,
                  'share_inside_pct':100*((data.ratio/median-1).abs() <= relative_band).mean()})
bands = pd.DataFrame(bands)
display(bands)
bands.to_csv(ARTIFACTS / 'w70_f30_ratio_relative_bands.csv', index=False)

Центр распределения очень узкий: 89.6% рабочих точек находятся в пределах ±1% от медианы, 99.1% — в пределах ±2%. Центральные 90% лежат примерно в диапазоне 0.773–0.789.

## Изменение за 10 минут

In [ ]:
central_ratio = raw.ratio.where(valid_mask & raw.ratio.between(p01,p99))
pair_mask = central_ratio.notna() & central_ratio.shift().notna()
absolute_change = central_ratio.diff().abs()[pair_mask]
relative_change = (central_ratio.diff()/central_ratio.shift()).abs()[pair_mask]
change = pd.DataFrame({
    'quantile':[.5,.9,.95,.99,.999],
    'absolute_change':[absolute_change.quantile(q) for q in [.5,.9,.95,.99,.999]],
    'relative_change_pct':[100*relative_change.quantile(q) for q in [.5,.9,.95,.99,.999]],
})
display(change)
change.to_csv(ARTIFACTS / 'w70_f30_ratio_10min_changes.csv', index=False)
px.line(change, x='quantile', y='relative_change_pct', markers=True,
        title='W70/F30: квантиль изменения за 10 минут, %').show()

Медианное изменение за 10 минут — около 0.27%; p95 — 1.19%; p99 — 1.61%. Резкие изменения больше 2% встречаются примерно в 0.1% центральных рабочих пар.

## Дрейф по годам и месяцам

In [ ]:
data['year'] = data.date.dt.year
yearly = data.groupby('year').ratio.agg(
    n='size', mean='mean', std='std', median='median',
    p01=lambda s:s.quantile(.01), p05=lambda s:s.quantile(.05),
    p95=lambda s:s.quantile(.95), p99=lambda s:s.quantile(.99),
).reset_index()
monthly = data.set_index('date').ratio.resample('MS').agg(
    count='size', mean='mean', median='median', std='std',
    p01=lambda s:s.quantile(.01), p99=lambda s:s.quantile(.99),
).reset_index()
display(yearly, monthly.nsmallest(8,'median'), monthly.nlargest(8,'median'))
yearly.to_csv(ARTIFACTS / 'w70_f30_ratio_yearly.csv', index=False)
monthly.to_csv(ARTIFACTS / 'w70_f30_ratio_monthly.csv', index=False)
px.line(monthly, x='date', y='median', error_y=monthly.p99-monthly['median'],
        error_y_minus=monthly['median']-monthly.p01,
        title='W70/F30: месячная медиана и диапазон p01–p99').show()

Годовая медиана снизилась с 0.7836 в 2023 до 0.7801 в 2026 — примерно на 0.45%. Наибольшая месячная медиана 0.7893 была в июне 2023. Минимум 0.7738 в апреле 2024 совпадает с нарушенным/остановочным периодом и не должен интерпретироваться как нормальный дрейф.

## С чем изменяется отношение

In [ ]:
analysis = central.drop(columns=['year'], errors='ignore')
level = analysis.drop(columns='date').corr(method='spearman').ratio.drop('ratio')
change_corr = analysis.drop(columns='date').diff().corr(method='spearman').ratio.drop('ratio')
correlations = pd.DataFrame({'level_spearman':level, 'change_spearman':change_corr}).rename_axis('tag').reset_index()
correlations['abs_level'] = correlations.level_spearman.abs()
correlations = correlations.sort_values('abs_level', ascending=False)
display(correlations.head(20))
correlations.to_csv(ARTIFACTS / 'w70_f30_ratio_correlations.csv', index=False)
px.bar(correlations.head(18), x='level_spearman', y='tag', orientation='h',
       color='change_spearman', title='W70/F30: связи с параметрами АВТ').show()

Наиболее заметные уровневые связи: T71 +0.37, F69 −0.36, T66 +0.32, F63 +0.27 и несколько температурных тегов около +0.23…+0.25. Это умеренные режимные связи. Изменение самого отношения слабо связано с изменениями большинства внешних параметров. Сильная отрицательная связь Δratio с ΔF30 частично механическая, потому что F30 находится в знаменателе.

## Вывод

В нормальном режиме W70/F30 меняется мало: типичный robust CV около 0.60%, p05–p95 занимает около 2.03% медианы, p01–p99 — около 3.24%. Наблюдаемый медленный дрейф между годами меньше 0.5%, тогда как остановы и кодовые значения дают изменения на порядок больше.

Для контроля данных разумно использовать динамический baseline отношения: rolling median за 7–30 дней и отклонение в robust sigma/MAD. Постоянные фиксированные границы можно применять только как исследовательские, а не как технологические limits.